Assignment 1 (Image Stitching) - Matteo Michetti - ID 6457183

In [ ]:
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from skimage.feature import corner_harris, corner_peaks
from skimage.transform import AffineTransform


In [ ]:
img1 = cv2.imread("")
img2 = cv2.imread("")

# FIX: cv2 legge in BGR, matplotlib vuole RGB
img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(12, 5))
plt.subplot(121); plt.imshow(img1_rgb); plt.title("Image 1"); plt.axis('off')
plt.subplot(122); plt.imshow(img2_rgb); plt.title("Image 2"); plt.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
coords1 = corner_peaks(corner_harris(img1_gray), min_distance=5, threshold_rel=0.001)
coords2 = corner_peaks(corner_harris(img2_gray), min_distance=5, threshold_rel=0.001)

plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.imshow(img1_gray, cmap='gray')
plt.plot(coords1[:, 1], coords1[:, 0], '+r', markersize=4)  # col=x, row=y
plt.title(f'Harris Key-points Img1: {len(coords1)}')
plt.subplot(122)
plt.imshow(img2_gray, cmap='gray')
plt.plot(coords2[:, 1], coords2[:, 0], '+r', markersize=4)
plt.title(f'Harris Key-points Img2: {len(coords2)}')
plt.tight_layout(); plt.show()


In [ ]:
sift = cv2.SIFT_create()
keypoints1, descriptors1 = sift.detectAndCompute(img1_gray, None)
keypoints2, descriptors2 = sift.detectAndCompute(img2_gray, None)

img_sift1 = cv2.drawKeypoints(img1_gray, keypoints1, None,
                               flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img_sift2 = cv2.drawKeypoints(img2_gray, keypoints2, None,
                               flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(12, 5))
plt.subplot(121); plt.imshow(img_sift1, cmap='gray'); plt.title('SIFT Img1'); plt.axis('off')
plt.subplot(122); plt.imshow(img_sift2, cmap='gray'); plt.title('SIFT Img2'); plt.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
#NCC
PATCH_RADIUS = 7

def extract_patches(coords, img_gray, r):
    patches = []
    valid_coords = []
    h, w = img_gray.shape
    for pt in coords:
        row, col = pt
        # Boundary checks are essential here to prevent index out-of-bounds errors
        # when points are located too close to the image edges.
        if row - r < 0 or row + r + 1 > h or col - r < 0 or col + r + 1 > w:
            continue
        patch = img_gray[row - r: row + r + 1, col - r: col + r + 1].astype(float)
        patch_f = patch.flatten()
        mean, std = np.mean(patch_f), np.std(patch_f)
        #check to skip black zones
        if std > 0:
            patch_f = (patch_f - mean) / std
        patches.append(patch_f)
        valid_coords.append(pt)
    return patches, np.array(valid_coords)

patches1, vcoords1 = extract_patches(coords1, img1_gray, PATCH_RADIUS)
patches2, vcoords2 = extract_patches(coords2, img2_gray, PATCH_RADIUS)
# Matrix NCC (i x j)
ncc_mat = np.zeros((len(patches1), len(patches2)))

patch_len = (2 * PATCH_RADIUS + 1) ** 2

# matrix construction for ncc
mat1 = np.array(patches1)
mat2 = np.array(patches2)

# vectorization via dot product

ncc_mat = np.dot(mat1, mat2.T) / patch_len
# matching selection defining a treshold
NCC_THRESHOLD = 0.5
listindex_ncc = []
for i in range(ncc_mat.shape[0]):
    best_j = np.argmax(ncc_mat[i, :])
    if ncc_mat[i, best_j] > NCC_THRESHOLD:
        listindex_ncc.append((i, best_j))

print(f"Matches NCC found: {len(listindex_ncc)}")


In [ ]:
#EUCL
def extract_patches_l2(coords, img_gray, r):
    patches = []
    valid_coords = []
    h, w = img_gray.shape
    for pt in coords:
        row, col = pt
        if row - r < 0 or row + r + 1 > h or col - r < 0 or col + r + 1 > w:
            continue
        patch = img_gray[row - r: row + r + 1, col - r: col + r + 1].astype(float)
        patch_f = patch.flatten()
        norm = np.linalg.norm(patch_f)
        patch_f = patch_f / norm if norm > 0 else patch_f
        patches.append(patch_f)
        valid_coords.append(pt)
    return patches, np.array(valid_coords)

patches1_l2, vcoords1_l2 = extract_patches_l2(coords1, img1_gray, PATCH_RADIUS)
patches2_l2, vcoords2_l2 = extract_patches_l2(coords2, img2_gray, PATCH_RADIUS)

P1 = np.array(patches1_l2)
P2 = np.array(patches2_l2)

# Vectorized Euclidean distance calculation.
dist_mat = np.sqrt(
    np.sum(P1 ** 2, axis=1, keepdims=True)
    + np.sum(P2 ** 2, axis=1)
    - 2 * (P1 @ P2.T)
)
#Filtering matches based on Euclidean distance, I pick iff below trshold
DIST_THRESHOLD = 0.5
listindex_euc = []
for i in range(dist_mat.shape[0]):
    best_j = np.argmin(dist_mat[i, :])
    if dist_mat[i, best_j] < DIST_THRESHOLD:
        listindex_euc.append((i, best_j))

print(f"Match Euclidean found: {len(listindex_euc)}")

#choose if use the ncc or the Euclidean,
# NCC = listindex_ncc src = vcoors1 and dst =vcoords2
# for the Euclidean listindex is listindex_euc and src = vcoords1_l2 and dst = vcoords2_l2
listindex = listindex_ncc
vcoords_src = vcoords1 # coordinates
vcoords_dst = vcoords2

In [ ]:
#RANSAC
N_ITERATIONS = 2000
RANSAC_THRESHOLD = 7.0   # max distance to be selected as inlier
N_SAMPLES = 3 # number of points to estimate affine transformation

best_inliers_count = 0
best_transform = None
best_inliers_list = []


for index in range(N_ITERATIONS):
    #random extraction
    sample = random.sample(listindex, N_SAMPLES)

    # flipping coords for the affinetransformation
    src_pts = np.array([[vcoords_src[i][1], vcoords_src[i][0]] for i, j in sample])
    dst_pts = np.array([[vcoords_dst[j][1], vcoords_dst[j][0]] for i, j in sample])

    # estimation
    tform = AffineTransform()
    tform.estimate(src_pts, dst_pts)
    M = tform.params

    #count how many matches satisfy the model
    current_inliers = []
    for i, j in listindex:
        pt_src = np.array([vcoords_src[i][1], vcoords_src[i][0], 1.0])
        pt_dst_expected = np.dot(M, pt_src)

        #Euclidean distance to verify consistency
        dist = np.linalg.norm(pt_dst_expected[:2] - np.array([vcoords_dst[j][1], vcoords_dst[j][0]]))

        if dist < RANSAC_THRESHOLD:
            current_inliers.append((i, j))

    # save best model
    if len(current_inliers) > best_inliers_count:
        best_inliers_count = len(current_inliers)
        best_inliers_mask = current_inliers
        best_transform = M

#
total = len(listindex)
outliers = total - best_inliers_count
print(f"RANSAC"
      f" Inlier: {best_inliers_count} / {total}  |  Outlier: {outliers}")

# inliers average residual
residuals = []
for i, j in best_inliers_mask:
    x1, y1 = vcoords_src[i][1], vcoords_src[i][0]
    x2, y2 = vcoords_dst[j][1], vcoords_dst[j][0]
    pt_t = best_transform @ np.array([x1, y1, 1.0])
    residuals.append((pt_t[0] - x2) ** 2 + (pt_t[1] - y2) ** 2)
print(f"Average residual: {np.mean(residuals):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img1_gray, cmap='gray')
axes[1].imshow(img2_gray, cmap='gray')
for i, j in best_inliers_mask:
    r1, c1 = vcoords_src[i]
    r2, c2 = vcoords_dst[j]
    axes[0].plot(c1, r1, 'go', markersize=4)
    axes[1].plot(c2, r2, 'go', markersize=4)
axes[0].set_title(f'Inlier in Img1 ({best_inliers_count})')
axes[1].set_title(f'Inlier in Img2 ({best_inliers_count})')
plt.tight_layout(); plt.show()


In [ ]:
#wrapping

M = best_transform.astype(np.float32)

h1, w1 = img1.shape[:2]
h2, w2 = img2.shape[:2]

#calculate where the corners of img1 end up after the affine transformation
corners1 = np.array([[0, 0, 1],
                      [w1, 0, 1],
                      [0, h1, 1],
                      [w1, h1, 1]], dtype=float).T
corners1_t = (M @ corners1).T
xs = corners1_t[:, 0]
ys = corners1_t[:, 1]

#Determine the global canvas size
x_min = min(0, xs.min())
y_min = min(0, ys.min())
x_max = max(w2, xs.max())
y_max = max(h2, ys.max())

canvas_w = int(np.ceil(x_max - x_min))
canvas_h = int(np.ceil(y_max - y_min))

#apply a translation offset to bring the entire panorama into the positive coordinate space.
T_offset = np.array([[1, 0, -x_min],
                      [0, 1, -y_min],
                      [0, 0,  1   ]], dtype=np.float32)

M_final = T_offset @ M

# Warp img1
panorama = cv2.warpPerspective(img1, M_final, (canvas_w, canvas_h))


ox, oy = int(-x_min), int(-y_min)
panorama[oy:oy + h2, ox:ox + w2] = img2

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
plt.title(f"Final panorama Inlier RANSAC: {best_inliers_count}")
plt.axis('off')
plt.tight_layout()
plt.show()

cv2.imwrite("", panorama)

